# Pathshala — Notebook 1: SSC books

This notebook OCRs and indexes `ssc-2026-bangla-1.pdf` and `ssc-2026-bangla-2.pdf`. It exports two artifacts for the second notebook: `ssc-indexes.zip` and `ssc-ocr-checkpoints.zip`.

Attach a private Kaggle dataset containing the two SSC PDFs (the dataset may also contain the HSC PDFs), and attach this repository as a Kaggle dataset or upload `kaggle/build_indexes.py`. Use a GPU accelerator and enable Internet for the first model download. Keep the PDFs, OCR text, and ZIP files private.

In [ ]:
!nvidia-smi


In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-ben


In [ ]:
!python -m pip install -q --upgrade-strategy only-if-needed pymupdf==1.26.4 'sentence-transformers>=5.1.1,<6.0.0' tqdm==4.67.1


In [ ]:
!python -m pip install -q virtualenv
!python -m virtualenv --clear --system-site-packages /kaggle/working/paddle-env
!/kaggle/working/paddle-env/bin/python -m pip install -q --upgrade pip setuptools wheel wrapt
!/kaggle/working/paddle-env/bin/python -m pip install -q paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/
!/kaggle/working/paddle-env/bin/python -m pip install -q 'paddleocr[doc-parser]>=3.6.0,<3.7.0' pymupdf==1.26.4 'numpy>=1.26,<3' tqdm==4.67.1


In [ ]:
import subprocess
import torch

print('Torch:', torch.__version__)
print('Torch CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before continuing'
paddle_check = (
    'import paddle; '
    'print("Paddle:", paddle.__version__); '
    'print("Paddle compiled with CUDA:", paddle.device.is_compiled_with_cuda()); '
    'print("Paddle device:", paddle.device.get_device()); '
    'assert paddle.device.is_compiled_with_cuda(); paddle.utils.run_check()'
)
subprocess.run(['/kaggle/working/paddle-env/bin/python', '-c', paddle_check], check=True)
print('Torch and Paddle runtimes are ready.')


In [ ]:
from pathlib import Path
import shutil

candidates = list(Path('/kaggle/input').rglob('build_indexes.py'))
print('Indexer candidates:')
for path in candidates:
    print(' -', path)
assert candidates, (
    'No build_indexes.py found. Attach the repository as a Kaggle dataset '
    'or upload kaggle/build_indexes.py.'
)
source = next((path for path in candidates if path.parent.name == 'kaggle'), candidates[0])
shutil.copy2(source, '/kaggle/working/build_indexes.py')
print('Copied:', source)


In [ ]:
from pathlib import Path

SSC_IDS = ['ssc-2026-bangla-1', 'ssc-2026-bangla-2']
pdf_paths = {}
for book_id in SSC_IDS:
    matches = list(Path('/kaggle/input').rglob(f'{book_id}.pdf'))
    print(book_id, matches)
    assert len(matches) == 1, f'Expected exactly one {book_id}.pdf'
    assert matches[0].stat().st_size <= 100 * 1024 * 1024, f'{book_id}.pdf is over 100 MB'
    pdf_paths[book_id] = matches[0]
print('SSC input PDFs are ready.')


## OCR the SSC group

The command is page-checkpointed. If a Kaggle cell stops, rerun the same cell in the same session; completed pages are reused.

In [ ]:
!PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK=True /kaggle/working/paddle-env/bin/python /kaggle/working/build_indexes.py --stage ocr --book ssc --ocr-engine paddle-vl --ocr-workers 1 --dpi 250


In [ ]:
from pathlib import Path
import json
import pymupdf

ocr_root = Path('/kaggle/working/pathshala-build/ocr')
review_pages = []
total = 0
for book_id in SSC_IDS:
    with pymupdf.open(pdf_paths[book_id]) as pdf:
        expected = len(pdf)
    checkpoints = sorted((ocr_root / book_id).glob('*.json'))
    print(book_id, 'checkpoints:', len(checkpoints), 'expected:', expected)
    assert len(checkpoints) == expected, f'{book_id} OCR is incomplete'
    total += len(checkpoints)
    for checkpoint in checkpoints:
        record = json.loads(checkpoint.read_text(encoding='utf-8'))
        if record.get('status') != 'ok':
            review_pages.append((book_id, record.get('page')))
print('Total SSC OCR checkpoints:', total, 'expected:', 484)
print('Automatically flagged pages:', review_pages)
assert total == 484, 'The two SSC books should contain 484 pages'


## Build the SSC embedding indexes

This stage runs in Kaggle's normal PyTorch environment and uses `Qwen/Qwen3-Embedding-0.6B` on the GPU.

In [ ]:
!python /kaggle/working/build_indexes.py --stage index --book ssc --ocr-engine paddle-vl --ocr-workers 1 --embed-batch-size 16 --dpi 250


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import shutil
import zipfile

archive = Path('/kaggle/working/pathshala-indexes.zip')
assert archive.is_file() and zipfile.is_zipfile(archive), 'SSC index ZIP was not created'
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None, 'The SSC ZIP is corrupt'
    names = bundle.namelist()
    for book_id in SSC_IDS:
        assert any(f'indexes/{book_id}/active.json' in name for name in names), f'Missing {book_id}'

ssc_archive = Path('/kaggle/working/ssc-indexes.zip')
shutil.copy2(archive, ssc_archive)
ocr_archive = Path(shutil.make_archive('/kaggle/working/ssc-ocr-checkpoints', 'zip', root_dir='/kaggle/working/pathshala-build/ocr', base_dir='.'))
print('Download both artifacts before starting Notebook 2:')
print(' -', ssc_archive)
print(' -', ocr_archive)
display(FileLink(str(ssc_archive)))
display(FileLink(str(ocr_archive)))
